# 28 · DLM em painel com AIC/BIC e FE/RE

32 aplicações principais previstas, 29 estimáveis. Grade K=1..24, amostra comum de 24 meses e 27 UFs; F/LM/Hausman, cinco hipóteses e sensibilidade de covariância.

## 1. Entrada, integridade e objetivos

Os códigos desta revisão não substituem os arquivos originais; os dados consolidados devem ser fornecidos localmente. A inferência é exploratória.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
SAIDA=ROOT/'outputs/tables/revisao_estadual_20261019'
assert SAIDA.exists()
print('Repositório:',ROOT)
print('Base:',os.getenv('TCC_BASE_CSV',str(ROOT/'data/processed/df_tcc_2013_2024.csv')))


Repositório: /mnt/data/tcc_revisao/repo
Base: /mnt/data/tcc_bases/df_tcc_2013_2024_recovered.csv


## 2. Especificação e seleção

`Δlog(I_it)=α_i+λ_t+Σ_{k=0}^K β_k Δlog(1+D_{i,t-k})+ε_it`, com TWFE por UF e mês-ano. AIC e BIC calculados para cada K em amostra comum, com o total de parâmetros incluindo os efeitos fixos. Modelos RE/pooled são diagnósticos clássicos comparativos.

In [2]:
from analise_estadual import load,dlm
res=dlm(load())
sel=pd.read_csv(SAIDA/"dlm_selecao.csv")
diag=pd.read_csv(SAIDA/"dlm_diagnosticos_fe_re.csv")
print("hipóteses calculadas:",len(res),"tipologias estimáveis:",len(diag))

hipóteses calculadas: 1160 tipologias estimáveis: 29


## 3. Seleção do horizonte

In [3]:
comparacao=sel.pivot_table(index=["periodo","tipologia"],columns="criterio",values="K").reset_index()
display(comparacao)
print("Divergências de K:",int((comparacao.AIC!=comparacao.BIC).sum()))

criterio,periodo,tipologia,AIC,BIC
0,Pré,tipo_alagamentos,1.0,1.0
1,Pré,tipo_chuvas_intensas,15.0,1.0
2,Pré,tipo_doencas_infecciosas,1.0,1.0
3,Pré,tipo_enxurradas,1.0,1.0
4,Pré,tipo_erosao,1.0,1.0
5,Pré,tipo_estiagem_e_seca,1.0,1.0
6,Pré,tipo_granizo,1.0,1.0
7,Pré,tipo_incendio_florestal,1.0,1.0
8,Pré,tipo_inundacoes,1.0,1.0
9,Pré,tipo_movimento_de_massa,1.0,1.0


Divergências de K: 8


## 4. Inferência usual e proteção conservadora da busca de K

In [4]:
resumo=res.groupby(["periodo","criterio","covariancia"])[["rejeicao_bh","rejeicao_bh_grade"]].sum()
display(resumo)
display(res[(res.criterio=="BIC")&(res.covariancia=="dk_regra")&(res.rejeicao_bh_grade)])

rejeicao_bh  rejeicao_bh_grade
periodo criterio covariancia                                
Pré     AIC      cluster_uf             3                  1
                 dk12                   4                  4
                 dk6                    4                  4
                 dk_regra               4                  4
        BIC      cluster_uf             0                  0
                 dk12                   2                  0
                 dk6                    1                  0
                 dk_regra               0                  0
Total   AIC      cluster_uf            25                 10
                 dk12                  33                 14
                 dk6                   32                  2
                 dk_regra              31                  0
        BIC      cluster_uf             9                  0
                 dk12                  20                  3
                 dk6                   18                  0
                 dk_regra              19                  0

,periodo,tipologia,criterio,K,covariancia,endpoint,gl_residuais,p,estimate,se,ic_inf,ic_sup,estatistica,df_num,status,q_bh,rejeicao_nominal,rejeicao_bh,p_grade_bonferroni,q_bh_grade,rejeicao_bh_grade


## 5. F, LM e Hausman e dependência entre UFs

In [5]:
display(diag[["periodo","tipologia","p_F","p_LM","p_Hausman","media_absoluta","n_UFs_LB_p05"]]);print("Hausman disponível:",diag.p_Hausman.notna().sum())

,periodo,tipologia,p_F,p_LM,p_Hausman,media_absoluta,n_UFs_LB_p05
0,Pré,tipo_alagamentos,0.999889,1.0,0.556612,0.189883,12
1,Pré,tipo_chuvas_intensas,0.999893,1.0,0.703192,0.190910,12
2,Pré,tipo_doencas_infecciosas,0.999890,1.0,NaN,0.190245,13
3,Pré,tipo_enxurradas,0.999889,1.0,0.694005,0.190048,12
4,Pré,tipo_erosao,0.999893,1.0,0.784738,0.190287,13
5,Pré,tipo_estiagem_e_seca,0.999891,1.0,0.987544,0.190734,12
6,Pré,tipo_granizo,0.999890,1.0,0.985056,0.190370,12
7,Pré,tipo_incendio_florestal,0.999891,1.0,NaN,0.191500,13
8,Pré,tipo_inundacoes,0.999886,1.0,0.542312,0.189586,13
9,Pré,tipo_movimento_de_massa,0.999893,1.0,0.758409,0.190586,12


Hausman disponível: 22


## 6. Interpretação

A seleção BIC K=1 é uma preferência pela borda inferior da grade, não prova ausência de associação após um mês. A proteção por busca K e BH não é inferência seletiva exata; os Wald conjuntos com muitos lags podem ser numericamente frágeis. Inferência condicionada à covariância e diagnóstico residual; não causal.